<a href="https://colab.research.google.com/github/IgnBys/3D_splatting/blob/main/gaussian_splatting_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3DGS in colab

This is slightly modified version of github.com/camenduru/gaussian-splatting-colab, with fixed dependencies and C files for CUDA 12.8 version

## Installing all the dependencies and wheels

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile ninja

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 25.50 MiB/s, done.
Resolving deltas: 100% (346/346), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 (delta 148), pack-reused 2971 (from 1)        
Receiving objects:

In [ ]:
%cd /content/gaussian-splatting

/content/gaussian-splatting


In [ ]:
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization

  Preparing metadata (setup.py) ... done


In [ ]:
!nvcc --version

import torch
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
PyTorch version: 2.10.0+cu128
PyTorch CUDA version: 12.8


In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'

In [ ]:
# The compiler is complaining that FLT_MAX is undefined.
# In newer versions of CUDA and GCC, certain headers like <float.h> are no longer included automatically,
# so we manually insert the missing header into the source code and then reinstall simple_knn.
!sed -i '1i #include <float.h>' /content/gaussian-splatting/submodules/simple-knn/simple_knn.cu

%cd /content/gaussian-splatting
!pip install ./submodules/simple-knn

/content/gaussian-splatting
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3550105 sha256=0fac6ebc8583b5d6154ede0a489c1a70fa304b97fee4d69b50583af5a190f876
  Stored in directory: /root/.cache/pip/wheels/0a/f2/1b/255828ebad94ea248378281b7926639d83ce4f394f0052800d
Successfully built simple_knn


## Download the sample scenes

## Run training of gaussian splats

## Run training for our custom data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, define your image path
# Example: IMAGE_PATH = '/content/drive/MyDrive/magisterka/ignat_photos'
import os
print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [2]:
import os
%cd /content/gaussian-splatting
print(f"Current working directory: {os.getcwd()}")

[Errno 2] No such file or directory: '/content/gaussian-splatting'
/content
Current working directory: /content


In [6]:
import os
import shutil
import glob

# Define the paths
SOURCE_PHOTOS = '/content/drive/MyDrive/magisterka/ignat_photos'
REPO_DATA_PATH = '/content/gaussian-splatting/data/ignat_photos'
REPO_INPUT_PATH = os.path.join(REPO_DATA_PATH, 'input')

# 1. Clear and recreate the input directory to ensure a clean state
if os.path.exists(REPO_INPUT_PATH):
    shutil.rmtree(REPO_INPUT_PATH)
os.makedirs(REPO_INPUT_PATH, exist_ok=True)

# 2. Copy all photos recursively from Drive to the local data/input folder
print(f"Searching for photos in {SOURCE_PHOTOS}...")
extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
found_files = []
for ext in extensions:
    found_files.extend(glob.glob(os.path.join(SOURCE_PHOTOS, '**', ext), recursive=True))

print(f"Copying {len(found_files)} photos to {REPO_INPUT_PATH}...")
for i, file_path in enumerate(found_files):
    # Flatten the structure by using just the filename
    dest_path = os.path.join(REPO_INPUT_PATH, os.path.basename(file_path))
    shutil.copy(file_path, dest_path)



Searching for photos in /content/drive/MyDrive/magisterka/ignat_photos...
Copying 78 photos to /content/gaussian-splatting/data/ignat_photos/input...


In [4]:
SOURCE_PATH = '/content/drive/MyDrive/magisterka/ignat_photos'

if os.path.exists(SOURCE_PATH):
    photos = os.listdir(SOURCE_PATH)
    print(f'Successfully found folder with {len(photos)} items.')
else:
    print(f'Error: The path {SOURCE_PATH} does not exist. Please check the folder name.')

Successfully found folder with 2 items.


In [ ]:
# All of these is to use GPU-accelerated version of COLMAP

# Install xvfb for OpenGL virtual display and required dependencies to build COLMAP
!sudo apt-get update
!sudo apt-get install -y xvfb cmake ninja-build \
    libboost-program-options-dev libboost-filesystem-dev libboost-graph-dev \
    libboost-system-dev libboost-test-dev libsuitesparse-dev \
    libfreeimage-dev libgoogle-glog-dev libgflags-dev \
    libglew-dev libqt5opengl5-dev qt5-qmake qtbase5-dev libceres-dev \
    libflann-dev libcgal-dev libmetis-dev

# Build COLMAP from source for GPU (CUDA) support
!rm -rf /content/colmap
!git clone --branch 3.9 https://github.com/colmap/colmap.git /content/colmap
%cd /content/colmap
!mkdir build
%cd build
!cmake .. -GNinja -DCUDA_ENABLED=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=native
!ninja
!sudo ninja install

# Return to the gaussian-splatting directory
%cd /content/gaussian-splatting


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-s

In [ ]:
# Run COLMAP conversion using xvfb-run to allow OpenGL context creation
!xvfb-run python convert.py -s /content/gaussian-splatting/data/ignat_photos

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
  53  2.724840e+02    2.54e-02    5.68e+02   4.22e+00   6.10e-01  8.99e+04        1    2.96e-03    1.70e-01
  54  2.724592e+02    2.48e-02    5.75e+02   4.21e+00   6.11e-01  9.09e+04        1    2.94e-03    1.73e-01
  55  2.724350e+02    2.42e-02    5.81e+02   4.21e+00   6.11e-01  9.19e+04        1    2.88e-03    1.76e-01
  56  2.724114e+02    2.36e-02    5.88e+02   4.21e+00   6.11e-01  9.29e+04        1    2.86e-03    1.79e-01
  57  2.723883e+02    2.31e-02    5.94e+02   4.21e+00   6.11e-01  9.39e+04        1    3.14e-03    1.82e-01
  58  2.723657e+02    2.25e-02    5.99e+02   4.21e+00   6.12e-01  9.50e+04        1    3.15e-03    1.86e-01
  59  2.723438e+02    2.20e-02    6.05e+02   4.21e+00   6.12e-01  9.61e+04        1    2.97e-03    1.89e-01
  60  2.723223e+02    2.14e-02    6.10e+02   4.21e+00   6.12e-01  9.72e+04        1    3.01e-03    1.92e-01
  61  2.723014e+02    2.09e-02    6.15e+02   4.20e+00   6.12e-01  9.83e

In [ ]:
!python train.py -s /content/gaussian-splatting/data/ignat_photos/

2026-05-10 22:04:15.755654: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/b109d513-f [10/05 22:04:21]
Reading camera 5/5 [10/05 22:04:21]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [10/05 22:04:21]
Loading Training Cameras [10/05 22:04:21]
Loading Test Cameras [10/05 22:04:21]
Number of points at initialisation :  605 [10/05 22:04:21]
Training progress:  23% 7000/30000 [00:58<03:46, 101.53it/s, Loss=0.0079387]
[ITER 7000] Evaluating train: L1 0.005939045455306768 PSNR 31.062673568725586 [10/05 22:05:25]

[ITER 7000] Saving Gaussians [10/05 22:05:25]
Training progress: 100% 30000/30000 [05:35<00:00, 89.49it/s, Loss=0.0038670]

[ITER 30000] Evaluating train: L1 0.0034140

# Save the results to Google Drive